# Silver - CETIC 2025

## Objetivo

Transformar os microdados brutos da TIC Domicílios 2025 em uma
estrutura analítica padronizada para o Radar Geracional de Conteúdo.

## Transformações

- seleção das variáveis relevantes;
- tratamento da faixa etária;
- conversão do peso amostral;
- padronização das respostas;
- transformação das categorias de colunas para linhas;
- identificação de respostas válidas e inválidas;
- padronização da nomenclatura das categorias.

## Categorias utilizadas no MVP

- Notícias
- Esportes
- Música
- Humor
- Games

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
df_cetic_bronze = spark.table(
    "workspace.mvp_bronze.cetic_individuos_2025"
)

print("Linhas Bronze:", df_cetic_bronze.count())
print("Colunas Bronze:", len(df_cetic_bronze.columns))

In [0]:
campos_cetic = [
    "FAIXA_ETARIA",
    "PESO",
    "TC4B_A",
    "TC4B_B",
    "TC4B_C",
    "TC4B_D",
    "TC4B_G"
]

for campo in campos_cetic:
    print(
        campo,
        "->",
        "OK" if campo in df_cetic_bronze.columns else "NÃO ENCONTRADO"
    )

In [0]:
for campo in [
    "FAIXA_ETARIA",
    "TC4B_A",
    "TC4B_B",
    "TC4B_C",
    "TC4B_D",
    "TC4B_G"
]:
    print(f"\n=== {campo} ===")

    df_cetic_bronze \
        .groupBy(campo) \
        .count() \
        .orderBy(campo) \
        .show(truncate=False)

In [0]:
display(
    df_cetic_bronze
    .select("PESO")
    .where(F.col("PESO").isNotNull())
    .limit(20)
)

In [0]:
df_cetic_base = (
    df_cetic_bronze
    .select(
        "FAIXA_ETARIA",
        "PESO",
        "TC4B_A",
        "TC4B_B",
        "TC4B_C",
        "TC4B_D",
        "TC4B_G"
    )
)

In [0]:
df_cetic_tipado = (
    df_cetic_base
    .withColumn(
        "faixa_etaria_codigo",
        F.col("FAIXA_ETARIA").cast("int")
    )
    .withColumn(
        "peso",
        F.regexp_replace(
            F.col("PESO"),
            ",",
            "."
        ).cast("double")
    )
)

In [0]:
df_cetic_tipado = (
    df_cetic_tipado
    .withColumn(
        "faixa_etaria",
        F.when(F.col("faixa_etaria_codigo") == 1, "10-15")
         .when(F.col("faixa_etaria_codigo") == 2, "16-24")
         .when(F.col("faixa_etaria_codigo") == 3, "25-34")
         .when(F.col("faixa_etaria_codigo") == 4, "35-44")
         .when(F.col("faixa_etaria_codigo") == 5, "45-59")
         .when(F.col("faixa_etaria_codigo") == 6, "60+")
         .otherwise("Não informado")
    )
)

In [0]:
df_cetic_long = (
    df_cetic_tipado
    .selectExpr(
        "faixa_etaria_codigo",
        "faixa_etaria",
        "peso",
        """
        stack(
            5,
            'Notícias', TC4B_A,
            'Esportes', TC4B_B,
            'Música', TC4B_C,
            'Humor', TC4B_D,
            'Games', TC4B_G
        ) as (categoria, resposta_original)
        """
    )
)

In [0]:
df_cetic_long = (
    df_cetic_long
    .withColumn(
        "resposta_codigo",
        F.col("resposta_original").cast("int")
    )
)

In [0]:
df_cetic_long = (
    df_cetic_long
    .withColumn(
        "resposta_valida",
        F.col("resposta_codigo").isin(0, 1)
    )
    .withColumn(
        "resposta",
        F.when(
            F.col("resposta_codigo") == 1,
            1
        )
        .when(
            F.col("resposta_codigo") == 0,
            0
        )
        .otherwise(F.lit(None).cast("int"))
    )
)

In [0]:
df_cetic_long = (
    df_cetic_long
    .withColumn(
        "status_resposta",
        F.when(F.col("resposta_codigo") == 1, "Sim")
         .when(F.col("resposta_codigo") == 0, "Não")
         .when(F.col("resposta_codigo") == 97, "Não sabe")
         .when(F.col("resposta_codigo") == 98, "Não respondeu")
         .when(F.col("resposta_codigo") == 99, "Não se aplica")
         .otherwise("Código não mapeado")
    )
)

In [0]:
df_cetic_silver = (
    df_cetic_long
    .withColumn("ano_referencia", F.lit(2025))
    .withColumn(
        "fonte",
        F.lit("CETIC.br - TIC Domicílios 2025")
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_cetic_silver = (
    df_cetic_silver
    .select(
        "ano_referencia",
        "faixa_etaria_codigo",
        "faixa_etaria",
        "categoria",
        "resposta_original",
        "resposta_codigo",
        "resposta",
        "resposta_valida",
        "status_resposta",
        "peso",
        "fonte",
        "_data_processamento"
    )
)

In [0]:
display(
    df_cetic_silver.limit(30)
)

In [0]:
print(
    "Linhas Silver:",
    df_cetic_silver.count()
)

In [0]:
display(
    df_cetic_silver
    .groupBy("categoria")
    .count()
    .orderBy("categoria")
)

In [0]:
display(
    df_cetic_silver
    .groupBy(
        "faixa_etaria_codigo",
        "faixa_etaria"
    )
    .count()
    .orderBy("faixa_etaria_codigo")
)

In [0]:
display(
    df_cetic_silver
    .groupBy(
        "resposta_codigo",
        "status_resposta"
    )
    .count()
    .orderBy("resposta_codigo")
)

In [0]:
display(
    df_cetic_silver
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("peso").isNull(), 1).otherwise(0)
        ).alias("peso_nulo"),
        F.min("peso").alias("peso_minimo"),
        F.max("peso").alias("peso_maximo")
    )
)

In [0]:
(
    df_cetic_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.mvp_silver.consumo_digital_cetic_2025"
    )
)